# Modulo 1 — AI Agents

**Caso pratico:** costruzione di un *Autonomous IT Operations Agent* capace di diagnosticare un incidente infrastrutturale usando strumenti Python.

L'obiettivo didattico non è semplicemente chiamare un LLM, ma comprendere il **ciclo agentico**:

```text
Observe → Decide → Act → Observe → ...
```

```text
            ┌──────────────── Observe (risultato del tool) ◄────────────────┐
            ▼                                                               │
   ┌─────────────────┐        ┌──────────────────┐        ┌────────────────────────┐
   │  Decide         │ ─────► │  Act             │ ─────► │  Environment           │
   │  policy π(a|h)  │        │  tool call       │        │  (server simulati)     │
   └─────────────────┘        └──────────────────┘        └────────────────────────┘
            │
            └──► Final answer  (condizione di terminazione)
```

### Obiettivi didattici

Al termine del modulo sarai in grado di:

1. distinguere **LLM**, **workflow** e **agente**;
2. progettare un **tool** come contratto software (input tipizzato → output strutturato);
3. implementare a mano un **agent loop** con stato, budget e condizione di terminazione;
4. applicare il principio di **minimo privilegio** e il pattern **human-in-the-loop**;
5. leggere una **trace** per ricostruire il ragionamento dell'agente;
6. riconoscere lo stesso ciclo dentro la **OpenAI Responses API** e l'**Agents SDK**.

### Struttura del notebook

Il notebook contiene tre livelli:

| Livello | Sezioni | API key | Concetto chiave |
|---|---|---|---|
| 1. Ambiente simulato e tools Python | 2 – 6 | No | Tool = contratto; separazione read / write |
| 2. Agent loop manuale con planner simulato | 7 – 11 | No | Loop, stato, trace, evidenza, approvazione umana |
| 3. Integrazione con OpenAI Responses API e Agents SDK | 12 – 13 | Sì (opzionale) | Il planner simulato viene sostituito da un LLM |
| Consolidamento | 14 – 15 | No | Challenge e debrief |

### Note operative

- Eseguire le celle **nell'ordine**: diverse celle leggono e modificano la variabile globale `servers` (stato dell'ambiente).
- Le celle della sezione 11 **modificano lo stato** (aggiungono righe di log). Per ripartire da zero usare `reset_environment()` (sezione 14) oppure *Kernel → Restart*.

## 1. Concetti chiave

Un Agent non coincide con il solo LLM. In questa lezione useremo la definizione operativa:

$$
\text{Agent} = \text{Model} + \text{Tools} + \text{State} + \text{ControlLoop}
$$

| Componente | Ruolo | Dove si trova nel notebook |
|---|---|---|
| **Model** | sceglie la prossima azione (o la risposta finale) | `simulated_planner` (sez. 7), poi un LLM (sez. 12–13) |
| **Tools** | unico canale con cui l'agente osserva e modifica il mondo | funzioni delle sez. 3–4 |
| **State** | memoria di lavoro: richiesta, osservazioni, risposta | dataclass `AgentState` (sez. 7) |
| **ControlLoop** | orchestra i turni, esegue i tool, applica budget e stop | `run_manual_agent` (sez. 8) |

### Formalizzazione

Il modello seleziona un'azione in funzione della storia disponibile:

$$
a_t \sim \pi(a \mid h_t)
$$

Dove:

- $a_t$ è l'azione scelta al tempo $t$;
- $h_t$ è la storia delle osservazioni e delle azioni precedenti;
- $\pi$ rappresenta, in modo concettuale, il criterio decisionale del sistema (la *policy*).

Il ciclo completo si scrive come una ricorrenza:

$$
\begin{aligned}
h_1 &= (u) &&\text{la storia iniziale contiene solo la richiesta utente } u\\
a_t &\sim \pi(a \mid h_t), \quad a_t \in \mathcal{A} = \mathcal{T} \cup \{\text{final}\} &&\text{azione: una tool call oppure la risposta finale}\\
o_t &= \text{Env}(a_t) &&\text{osservazione: risultato del tool}\\
h_{t+1} &= h_t \oplus (a_t, o_t) &&\text{la storia cresce per concatenazione}
\end{aligned}
$$

Il ciclo termina quando $a_t = \text{final}$ oppure quando $t = T_{\max}$ (budget esaurito).

- Con un **LLM**, $\pi$ è **stocastica**: l'output dipende dal campionamento (temperature, ecc.).
- Con il **planner simulato** della sez. 7, $\pi$ è **deterministica**: $a_t = f(h_t)$.

### Agente vs workflow

| | Workflow classico | Agente |
|---|---|---|
| Chi decide la sequenza di passi | il programmatore, a design-time | il modello, a runtime |
| Prevedibilità | alta | media / bassa |
| Gestione di casi imprevisti | solo quelli codificati | può adattarsi |
| Costo e latenza | bassi, stimabili | variabili (più chiamate al modello) |
| Auditabilità | semplice | richiede tracing (sez. 9) |

Questo schema *reason → act → observe* è quello reso popolare dal paradigma **ReAct** (Yao et al., 2022), in cui ragionamento e azioni si alternano.

### Setup: librerie utilizzate

Il laboratorio usa **solo la standard library** di Python, così da essere eseguibile ovunque senza installazioni. Le librerie `openai` e `openai-agents` servono solo nelle sezioni opzionali 12–13 e vengono importate lì, in modo *lazy*.

| Import | A cosa serve nel notebook |
|---|---|
| `from __future__ import annotations` | abilita la sintassi di tipo moderna (`str \| None`, `dict[str, Any]`) anche su Python < 3.10 |
| `json` | serializzare i risultati dei tool e leggere gli argomenti generati dal LLM |
| `os` | leggere le variabili d'ambiente `OPENAI_API_KEY` e `OPENAI_MODEL` |
| `deepcopy` | creare una copia indipendente dello stato iniziale (reset dell'ambiente) |
| `dataclass`, `field` | definire le strutture `ToolCall` e `AgentState` |
| `Any`, `Callable` | type hint per dizionari eterogenei e per le funzioni del registry |

In [ ]:
# Rende "lazy" la valutazione delle annotazioni di tipo (PEP 563): i type hint
# vengono memorizzati come stringhe e non valutati a runtime. Permette di usare
# `str | None` e `dict[str, Any]` anche su versioni di Python precedenti alla 3.10.
# Deve essere la prima istruzione del modulo/cella.
from __future__ import annotations

# json: serializzazione (dumps) e deserializzazione (loads) dei dati scambiati con il LLM
import json
# os: accesso alle variabili d'ambiente (API key e nome del modello)
import os
# deepcopy: copia ricorsiva di strutture annidate (dict di dict di liste)
from copy import deepcopy
# dataclass: genera automaticamente __init__, __repr__, __eq__ per classi "contenitore dati"
# field: permette di configurare i singoli attributi (es. default_factory per le liste)
from dataclasses import dataclass, field
# Any: valore di tipo qualsiasi; Callable: tipo "funzione invocabile"
from typing import Any, Callable
# load_dotenv: per importare API_KEY presenti in .env
from dotenv import load_dotenv

# Conferma visiva che tutte le import sono andate a buon fine
print("Ambiente pronto")

Ambiente pronto


## 2. Ambiente simulato

L'**ambiente** (*environment*) è tutto ciò con cui l'agente interagisce ma che **non controlla direttamente**: può solo osservarlo e modificarlo attraverso i tool. In termini di reinforcement learning corrisponde allo *stato del mondo* $s$.

L'ambiente rappresenta tre server aziendali:

- `web-01`: frontend web;
- `api-01`: backend applicativo;
- `db-01`: database.

Non useremo macchine reali; il laboratorio deve essere:
- **riproducibile** (stesso input → stessa trace, utile per valutare), 
- **sicuro** (nessun rischio di danni in produzione)
- **deterministico** (niente rumore nelle metriche).

### Architettura e scenario dell'incidente

```text
  Utente ──► web-01 (nginx) ──► api-01 (api) ──► db-01 (mysql)
               sano               SINTOMO           CAUSA
                                  attese / timeout  pool connessioni 99/100
```

<br/>

| Server | Ruolo | CPU % | RAM % | Servizio | Segnale nei log | Metrica specifica | Lettura |
|---|---|---|---|---|---|---|---|
| `web-01` | frontend | 39 | 61 | nginx | solo `INFO` | — | sano |
| `api-01` | backend | 48 | 67 | api | `WARNING` attese DB, timeout | — | **sintomo** |
| `db-01` | database | 52 | 69 | mysql | `INFO database healthy` | 99 / 100 connessioni | **causa radice** |

Punti didattici da far emergere:

- **CPU e RAM sono normali ovunque**: le metriche "classiche" non bastano a trovare il problema.
- Il **sintomo** (timeout) si manifesta in un nodo diverso dalla **causa** (DB): serve ragionare sulla catena delle dipendenze.
- I log del DB dicono *healthy*: l'agente riceve **evidenze contraddittorie** e deve pesare le fonti.

### Schema dei dati

```text
servers : dict[str, ServerState]

ServerState = {
    "role":        str,                 # ruolo architetturale
    "status":      str,                 # "online" / "offline"
    "cpu":         int,                 # % di utilizzo
    "memory":      int,                 # % di utilizzo
    "services":    dict[str, str],      # nome servizio -> stato
    "logs":        list[str],           # dal più vecchio al più recente
    "connections": {"active": int, "max": int}   # SOLO per i server database
}
```

**NB - Nella prossima cella: Perché `deepcopy`?**<br>
- `INITIAL_SERVERS` è la *fixture* (stato iniziale di riferimento); 
- `servers` è lo stato mutabile su cui lavorano i tool. <br>

Una copia superficiale (`dict.copy()`) duplicherebbe solo il primo livello: le liste di log e i dizionari interni sarebbero **condivisi**, quindi un riavvio modificherebbe anche la fixture e il reset non funzionerebbe.

In [2]:
# ---------------------------------------------------------------------------
# STATO INIZIALE DELL'AMBIENTE (fixture)
# Dizionario annidato: nome del server -> stato del server.
# Contiene la "ground truth" dello scenario: la causa dell'incidente (pool di
# connessioni del DB quasi saturo) è inserita qui, ma l'agente potrà scoprirla
# SOLO attraverso i tool, senza accesso diretto a questa struttura.
# Convenzione Python: nome in MAIUSCOLO = costante, da non modificare.
# ---------------------------------------------------------------------------
INITIAL_SERVERS = {
    # --- Frontend web: sano. Metriche nella norma, solo log INFO ------------
    "web-01": {
        "role": "frontend",               # ruolo architetturale del nodo
        "status": "online",               # stato della macchina
        "cpu": 39,                        # utilizzo CPU in percentuale
        "memory": 61,                     # utilizzo RAM in percentuale
        "services": {"nginx": "running"}, # servizi ospitati: nome -> stato
        "logs": [                         # log in ordine cronologico (ultimo = più recente)
            "12:01 INFO static asset served",
            "12:02 INFO request completed",
            "12:03 INFO healthcheck ok",
        ],
    },
    # --- Backend API: qui compaiono i SINTOMI (attese verso DB e timeout) ---
    "api-01": {
        "role": "backend",
        "status": "online",
        "cpu": 48,                        # CPU normale: il backend non è sovraccarico...
        "memory": 67,                     # ...eppure è lento -> sta ASPETTANDO qualcun altro
        "services": {"api": "running"},
        "logs": [
            "12:01 INFO request accepted",
            "12:02 WARNING database request waiting",  # richieste in coda verso il DB
            "12:03 WARNING request timeout",           # sintomo percepito dall'utente
            "12:04 WARNING database request waiting",  # il pattern si ripete: problema persistente
        ],
    },
    # --- Database: CPU/RAM normali, log "healthy", ma pool quasi saturo -----
    "db-01": {
        "role": "database",
        "status": "online",
        "cpu": 52,
        "memory": 69,
        "services": {"mysql": "running"},
        "logs": [
            "12:01 INFO database healthy",     # evidenza fuorviante: il DB "si dichiara" sano
            "12:02 INFO slow query log empty", # esclude le query lente come causa
        ],
        # Chiave presente SOLO nei server database: 99 connessioni attive su 100.
        # Utilizzo = 99%  ->  è la vera causa radice (root cause) dell'incidente.
        "connections": {"active": 99, "max": 100},
    },
}

# Stato MUTABILE su cui operano i tool. deepcopy crea una copia indipendente a
# tutti i livelli di annidamento: le modifiche successive (es. restart_service
# che aggiunge righe di log) non toccano INITIAL_SERVERS, che resta disponibile
# per il reset dell'ambiente (sezione 14).
servers = deepcopy(INITIAL_SERVERS)

# Ultima espressione della cella: Jupyter ne visualizza automaticamente il valore
servers

{'web-01': {'role': 'frontend',
  'status': 'online',
  'cpu': 39,
  'memory': 61,
  'services': {'nginx': 'running'},
  'logs': ['12:01 INFO static asset served',
   '12:02 INFO request completed',
   '12:03 INFO healthcheck ok']},
 'api-01': {'role': 'backend',
  'status': 'online',
  'cpu': 48,
  'memory': 67,
  'services': {'api': 'running'},
  'logs': ['12:01 INFO request accepted',
   '12:02 WARNING database request waiting',
   '12:03 WARNING request timeout',
   '12:04 WARNING database request waiting']},
 'db-01': {'role': 'database',
  'status': 'online',
  'cpu': 52,
  'memory': 69,
  'services': {'mysql': 'running'},
  'logs': ['12:01 INFO database healthy', '12:02 INFO slow query log empty'],
  'connections': {'active': 99, 'max': 100}}}

## 3. Tools read-only

I primi strumenti leggono informazioni dall'ambiente. **Non producono side effect**: si possono chiamare quante volte si vuole, in qualunque ordine, ottenendo sempre lo stesso risultato a parità di stato (sono *idempotenti*). Per questo sono a **basso rischio** e un agente può usarli liberamente.

Un tool è un **contratto software**:

$$
\text{tool}: \underbrace{\text{argomenti tipizzati}}_{\text{prodotti dal modello}} \longrightarrow \underbrace{\text{dict JSON-serializzabile}}_{\text{osservazione } o_t}
$$

### Principi di progettazione applicati

| Principio | Come è implementato | Perché conta per un agente |
|---|---|---|
| Output strutturato | ogni tool restituisce un `dict` | il risultato va serializzato in JSON e rimandato al modello |
| *Envelope* uniforme | chiave `success` sempre presente | il modello (o il loop) distingue subito successo e fallimento |
| **Errori come dati, non eccezioni** | `{"success": False, "error": "..."}` | l'errore diventa un'**osservazione**: il loop non si interrompe e il modello può correggersi |
| Codici di errore *machine-readable* | `SERVER_NOT_FOUND`, `NOT_A_DATABASE_SERVER` | stringhe stabili, facili da interpretare e da testare |
| Output minimale | `get_metrics` restituisce solo CPU e RAM | meno token, meno rumore, meno rischio di esporre dati non necessari |
| Validazione dell'input | controllo `server not in servers` | gli argomenti sono generati da un modello: vanno trattati come **input non fidato** |
| Docstring descrittiva | `"""Returns ..."""` | alcuni SDK la usano come descrizione del tool per il modello (sez. 13) |

### Struttura dell'output

```text
successo:  {"success": True,  "server": <nome>, ...payload specifico del tool...}
errore:    {"success": False, "error": <CODICE_ERRORE>, "server": <nome>}
```

### Catalogo dei tool read-only

| Tool | Argomenti | Payload restituito | Quando usarlo |
|---|---|---|---|
| `get_server_status` | `server` | `role`, `status`, `services` | verificare se una macchina/servizio è attivo |
| `get_metrics` | `server` | `cpu`, `memory` | problemi di performance |
| `read_log` | `server`, `last_lines=5` | ultime righe di log | cercare errori, warning, timeout |
| `get_database_connections` | `server="db-01"` | `active`, `max`, `utilization` | sospetta saturazione del pool di connessioni |

La metrica calcolata da `get_database_connections` è:

$$
U = \frac{\text{active}}{\text{max}} \in [0, 1]
$$

In [3]:
# ===========================================================================
# TOOL READ-ONLY
# Tutti leggono la variabile GLOBALE `servers` al momento della chiamata:
# se `servers` viene riassegnata (reset, sez. 14) i tool vedono il nuovo stato.
# ===========================================================================

def get_server_status(server: str) -> dict[str, Any]:
    """Returns server status and running services."""
    # Validazione dell'input: il nome del server arriva dal modello e potrebbe
    # essere errato o inventato. Invece di sollevare KeyError restituiamo un
    # errore strutturato, che diventerà un'osservazione utile per l'agente.
    if server not in servers:
        return {"success": False, "error": "SERVER_NOT_FOUND", "server": server}

    # Alias locale per accorciare gli accessi successivi (stesso oggetto, non una copia)
    s = servers[server]
    # Restituisce SOLO i campi pertinenti alla domanda "il server/servizio è attivo?"
    return {
        "success": True,
        "server": server,
        "role": s["role"],
        "status": s["status"],
        "services": s["services"],
    }


def get_metrics(server: str) -> dict[str, Any]:
    """Returns current CPU and memory utilization for a server."""
    # Stessa guardia di validazione: contratto uniforme tra tutti i tool
    if server not in servers:
        return {"success": False, "error": "SERVER_NOT_FOUND", "server": server}

    s = servers[server]
    # Payload minimale: solo le due metriche di performance
    return {
        "success": True,
        "server": server,
        "cpu": s["cpu"],        # % utilizzo CPU
        "memory": s["memory"],  # % utilizzo RAM
    }


def read_log(server: str, last_lines: int = 5) -> dict[str, Any]:
    """Returns latest log entries for a server."""
    # `last_lines` ha un default: il modello può ometterlo
    if server not in servers:
        return {"success": False, "error": "SERVER_NOT_FOUND", "server": server}

    return {
        "success": True,
        "server": server,
        # Slicing con indice negativo: [-n:] restituisce gli ultimi n elementi.
        # Se la lista ha meno di n righe, restituisce semplicemente tutta la lista.
        # Limitare le righe = controllare quanti token finiscono nel contesto del modello.
        "logs": servers[server]["logs"][-last_lines:],
    }


def get_database_connections(server: str = "db-01") -> dict[str, Any]:
    """Returns active and max database connections."""
    # Prima guardia: il server deve esistere
    if server not in servers:
        return {"success": False, "error": "SERVER_NOT_FOUND", "server": server}
    # Seconda guardia, di "dominio": il server deve essere un database.
    # Un errore specifico insegna qualcosa al modello sulla topologia del sistema.
    if "connections" not in servers[server]:
        return {"success": False, "error": "NOT_A_DATABASE_SERVER", "server": server}

    c = servers[server]["connections"]
    # Grado di utilizzo del pool: U = active / max, valore in [0, 1]
    utilization = c["active"] / c["max"]
    return {
        "success": True,
        "server": server,
        "active": c["active"],
        "max": c["max"],
        "utilization": utilization,  # frazione (0.99), non percentuale
    }


# Smoke test: chiamiamo i tool a mano, come farebbe l'agente, per verificarne l'output
print(get_metrics("api-01"))                 # CPU/RAM del backend: nella norma
print(read_log("api-01", last_lines=2))      # ultime 2 righe: timeout + attesa DB
print(get_database_connections())            # usa il default server="db-01": utilization 0.99

{'success': True, 'server': 'api-01', 'cpu': 48, 'memory': 67}
{'success': True, 'server': 'api-01', 'logs': ['12:03 WARNING request timeout', '12:04 WARNING database request waiting']}
{'success': True, 'server': 'db-01', 'active': 99, 'max': 100, 'utilization': 0.99}


## 4. Tool con side effect

Il riavvio di un servizio **modifica lo stato dell'ambiente**: è un'azione di scrittura, con conseguenze potenzialmente non reversibili (richieste in corso interrotte, sessioni perse). Per questo motivo richiede **approvazione esplicita**.

Regola didattica:

```text
READ → basso rischio
WRITE / RESTART → richiede controllo
DELETE / PAY / SEND → alto rischio
```

| Classe di azione | Esempi | Reversibile? | Controllo consigliato |
|---|---|---|---|
| READ | leggere log, metriche, stato | sì (nessun effetto) | nessuno; solo logging |
| WRITE / RESTART | riavviare servizi, cambiare configurazioni | parzialmente | approvazione umana o policy automatiche |
| DELETE / PAY / SEND | cancellare dati, pagamenti, inviare email a clienti | no | approvazione umana obbligatoria + audit |

### Principi di sicurezza applicati in `restart_service`

- **Fail-closed**: `approved=False` è il valore di default. Se nessuno approva, l'azione *non* avviene.
- **Ordine dei controlli**: prima si verifica l'approvazione, poi la validità degli argomenti. Senza approvazione non si tocca nulla.
- **Audit trail**: ogni riavvio lascia una riga nei log (`restarted by AI operator`) → l'azione è tracciabile e attribuibile.
- **Risposta "requires_approval"**: invece di un semplice errore, il tool restituisce una *proposta* che un orchestratore può mostrare a un operatore.

```text
restart_service(server, service, approved)
        │
        ├── approved == False ─────────► {"requires_approval": True}   (nessun effetto)
        ├── server inesistente ────────► SERVER_NOT_FOUND
        ├── servizio inesistente ──────► SERVICE_NOT_FOUND
        └── altrimenti ───────────────► stato = "running" + riga di log  (SIDE EFFECT)
```

> ⚠️ **Nota di design.** Il flag `approved` è un **parametro della funzione stessa**. Se il tool viene esposto a un LLM con `approved` nello schema (sez. 6), il modello potrebbe impostarlo a `True` da solo, aggirando il controllo umano. In produzione l'approvazione deve essere verificata dall'**orchestratore** (fuori dal controllo del modello), ad esempio intercettando la tool call prima di eseguirla. Il tema viene ripreso nelle sezioni 6, 11 e 12.

> Nota sulla simulazione: tutti i servizi sono già `running`, quindi il riavvio cambia lo stato solo formalmente; l'effetto osservabile è la nuova riga di log.

In [4]:
def restart_service(server: str, service: str, approved: bool = False) -> dict[str, Any]:
    """Restarts a service. Requires explicit human approval."""
    # 1) GATE DI APPROVAZIONE (fail-closed): è il PRIMO controllo.
    #    Senza approvazione esplicita non si esegue nulla e si restituisce
    #    una "proposta" che segnala la necessità di un intervento umano.
    if not approved:
        return {
            "success": False,
            "requires_approval": True,
            # f-string: interpola server e service nel messaggio leggibile
            "message": f"Restart of {service} on {server} requires approval.",
        }

    # 2) Validazione: il server deve esistere
    if server not in servers:
        return {"success": False, "error": "SERVER_NOT_FOUND", "server": server}

    # 3) Validazione: il servizio deve essere ospitato su quel server
    #    (impedisce di "riavviare" servizi inventati dal modello)
    if service not in servers[server]["services"]:
        return {
            "success": False,
            "error": "SERVICE_NOT_FOUND",
            "server": server,
            "service": service,
        }

    # 4) SIDE EFFECT: modifica dello stato globale dell'ambiente.
    #    Il servizio viene impostato a "running"...
    servers[server]["services"][service] = "running"
    # ...e si aggiunge una riga di log: audit trail dell'azione compiuta dall'AI
    servers[server]["logs"].append(f"12:05 INFO {service} restarted by AI operator")

    # 5) Esito positivo strutturato
    return {
        "success": True,
        "server": server,
        "service": service,
        "status": "restarted",
    }

# Chiamata SENZA approvazione (approved=False per default):
# il risultato atteso è {"success": False, "requires_approval": True, ...}
# e lo stato dell'ambiente resta invariato.
restart_service("api-01", "api")

{'success': False,
 'requires_approval': True,
 'message': 'Restart of api on api-01 requires approval.'}

## 5. Tool registry

Il registro associa il nome del tool alla funzione Python effettiva.

$$
\text{ToolName} \rightarrow \text{Callable}
$$

Formalmente è una funzione di lookup dall'insieme dei nomi all'insieme delle funzioni: $R : \mathcal{N} \to \mathcal{F}$ ;

l'esecuzione di un'azione diventa:

$$
o_t = R(\text{name}_t)(\,\text{**args}_t\,)
$$

**Perché serve?** Un LLM non esegue codice: produce **testo** (un nome di tool e un JSON di argomenti).

Il registry è il *ponte* tra la decisione del modello e la funzione Python da invocare (pattern **dispatch table**).

```text
LLM:   {"name": "get_metrics", "arguments": {"server": "api-01"}}
            │
            ▼  lookup nel registry
TOOL_REGISTRY["get_metrics"]  ──►  get_metrics(server="api-01")  ──►  risultato (dict)
```

**Aspetto di sicurezza:** il registry è una **allowlist**. Il modello può invocare *solo* le funzioni registrate. Non si deve mai risolvere il nome con meccanismi generici come `globals()[name]`, `getattr(module, name)` o `eval`: un nome scelto dal modello potrebbe portare a eseguire qualunque funzione del programma.

In [5]:
# Dispatch table: nome (stringa, come lo produce il modello) -> funzione Python.
# Type hint: dizionario con chiavi str e valori "funzioni con qualunque firma
# (...) che restituiscono dict[str, Any]".
# NB: i valori sono gli OGGETTI funzione (senza parentesi), non i loro risultati.
TOOL_REGISTRY: dict[str, Callable[..., dict[str, Any]]] = {
    "get_server_status": get_server_status,
    "get_metrics": get_metrics,
    "read_log": read_log,
    "get_database_connections": get_database_connections,
    "restart_service": restart_service,   # unico tool con side effect
}

# Elenco dei nomi registrati: è l'insieme T delle azioni-tool disponibili all'agente
list(TOOL_REGISTRY.keys())

['get_server_status',
 'get_metrics',
 'read_log',
 'get_database_connections',
 'restart_service']

## 6. Tool schema per un LLM

Lo schema descrive al modello quali strumenti sono disponibili e con quali parametri.

Non è semplice documentazione: è un **contratto software**. Il modello vede *solo* lo schema, non il codice Python: nome, descrizione e parametri sono tutto ciò su cui basa la scelta del tool e la costruzione degli argomenti.

### Tre rappresentazioni dello stesso tool

```text
 ┌──────────────────────┐   ┌──────────────────────┐   ┌──────────────────────────┐
 │ Funzione Python      │   │ TOOL_REGISTRY        │   │ tools_schema             │
 │ (implementazione)    │◄──│ (dispatch)           │   │ (interfaccia per il LLM) │
 │ get_metrics(server)  │   │ "get_metrics" -> fn  │   │ name, description, JSON  │
 └──────────────────────┘   └──────────────────────┘   └──────────────────────────┘
          ▲                                                        │
          └────────── devono restare ALLINEATE (nome e parametri) ─┘
```

Mantenerle allineate a mano è fragile: gli SDK (sez. 13) generano lo schema direttamente da firma e docstring della funzione (*single source of truth*).

### Anatomia di una voce dello schema

| Campo | Significato |
|---|---|
| `type: "function"` | tipo di tool (function calling) |
| `name` | identificatore: deve coincidere con la chiave del `TOOL_REGISTRY` |
| `description` | spiega **cosa fa** e **quando usarlo**: è di fatto *prompt engineering* del tool |
| `parameters` | **JSON Schema** degli argomenti |
| `properties` | nome e tipo di ogni parametro |
| `required` | parametri che il modello deve sempre fornire |
| `additionalProperties: false` | vieta parametri non dichiarati |
| `strict: true` | chiede al provider di garantire che gli argomenti generati rispettino lo schema (*structured outputs*) |

Si noti come le descrizioni guidino il ragionamento: quella di `get_database_connections` suggerisce esplicitamente di usarlo *"when logs suggest DB waiting, timeout..."*, cioè collega il sintomo nei log al tool di verifica.

> ⚠️ **Nota tecnica — strict mode.** Con `"strict": true`, OpenAI richiede che `additionalProperties` sia `false` e che **ogni chiave presente in `properties` compaia anche in `required`**. Per mantenere il laboratorio semplice, nello schema della cella successiva chiediamo quindi al modello di passare esplicitamente anche parametri che nelle funzioni Python hanno un default, come `last_lines=5` e `server="db-01"`. In alternativa un parametro semanticamente opzionale può essere modellato come nullable (`"type": ["integer", "null"]`) e normalizzato lato Python.

> ⚠️ **Nota di sicurezza.** Lo schema di `restart_service` espone il parametro `approved` al modello (vedi sez. 4). L'istruzione "requires explicit human approval" nella description è solo un'indicazione testuale, non un vincolo: il controllo va spostato nell'orchestratore, oppure il tool va semplicemente **non esposto** (come fa la sez. 13).


In [6]:
# Lista di definizioni di tool nel formato "function" della OpenAI Responses API.
# Con strict=True:
#   1) additionalProperties deve essere False
#   2) TUTTE le chiavi presenti in "properties" devono comparire in "required".
#
# Per semplicità didattica rendiamo quindi espliciti tutti i parametri.
# I default Python continuano a esistere nelle funzioni, ma lo schema chiede
# al modello di fornire comunque il valore.

tools_schema = [
    # --- Tool 1: stato del server ---------------------------------------------
    {
        "type": "function",
        "name": "get_server_status",
        "description": (
            "Returns status, role and services for a server. "
            "Use it to check if a machine or service is available."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "server": {"type": "string"}
            },
            "required": ["server"],
            "additionalProperties": False,
        },
        "strict": True,
    },

    # --- Tool 2: metriche CPU/RAM ---------------------------------------------
    {
        "type": "function",
        "name": "get_metrics",
        "description": (
            "Returns current CPU and RAM utilization for a server. "
            "Use it to diagnose performance issues."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "server": {"type": "string"}
            },
            "required": ["server"],
            "additionalProperties": False,
        },
        "strict": True,
    },

    # --- Tool 3: lettura log --------------------------------------------------
    {
        "type": "function",
        "name": "read_log",
        "description": (
            "Returns recent log lines for a server. "
            "Use it to investigate errors, timeouts or warnings. "
            "Always provide last_lines; use 5 when no different amount is needed."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "server": {"type": "string"},
                "last_lines": {
                    "type": "integer",
                    "description": "Number of recent log lines to return. Use 5 by default."
                },
            },
            "required": ["server", "last_lines"],
            "additionalProperties": False,
        },
        "strict": True,
    },

    # --- Tool 4: connessioni al database -------------------------------------
    {
        "type": "function",
        "name": "get_database_connections",
        "description": (
            "Returns active and maximum database connections. "
            "Use it when logs suggest DB waiting, timeout or connection pool saturation. "
            "Always provide the database server name; normally use db-01 in this lab."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "server": {"type": "string"}
            },
            "required": ["server"],
            "additionalProperties": False,
        },
        "strict": True,
    },

    # --- Tool 5: riavvio servizio (SIDE EFFECT) -------------------------------
    # Lo schema è formalmente valido, ma NON verrà esposto nella demo API:
    # l'approvazione non deve essere auto-dichiarata dal modello.
    {
        "type": "function",
        "name": "restart_service",
        "description": (
            "Restarts a service on a server. "
            "This action has side effects and requires explicit human approval."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "server": {"type": "string"},
                "service": {"type": "string"},
                "approved": {"type": "boolean"},
            },
            "required": ["server", "service", "approved"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

# Catalogo sicuro per la demo reale via API:
# esclude il tool con side effect finché non implementiamo Human-in-the-loop.
tools_schema_safe = [
    tool for tool in tools_schema
    if tool["name"] != "restart_service"
]

print("Tool disponibili nella demo API:")
for tool in tools_schema_safe:
    print("-", tool["name"])


Tool disponibili nella demo API:
- get_server_status
- get_metrics
- read_log
- get_database_connections


In [7]:
# Validazione didattica locale dello schema strict:
# verifica che ogni proprietà sia inclusa in "required".
for tool in tools_schema_safe:
    props = set(tool["parameters"].get("properties", {}).keys())
    required = set(tool["parameters"].get("required", []))
    missing = props - required
    assert not missing, f"{tool['name']}: proprietà mancanti in required: {missing}"
    assert tool["parameters"].get("additionalProperties") is False

print("Schema strict locale: OK")


Schema strict locale: OK


## 7. Una versione senza API: planner simulato

Prima di usare un LLM reale implementiamo un **planner deterministico**.

Questo serve a isolare il concetto di *agent loop* dal provider AI: loop, stato, dispatch e trace funzionano identici; nelle sezioni 12–13 cambierà **solo** la funzione che decide la prossima azione.

### Dalla formalizzazione al codice

| Concetto formale | Implementazione |
|---|---|
| azione $a_t \in \mathcal{T}$ (tool call) | `ToolCall(name, arguments)` |
| azione $a_t = \text{final}$ | una `str` con la risposta finale |
| storia $h_t$ | `AgentState`: `user_request` + lista `observations` |
| policy $\pi(a \mid h_t)$ | `simulated_planner(state)` |

Il tipo di ritorno `ToolCall | str` codifica lo **spazio delle azioni** $\mathcal{A} = \mathcal{T} \cup \{\text{final}\}$: il loop distinguerà i due casi con `isinstance`.

### La policy come albero di decisione

La policy è deterministica: dato un piano fisso $P = (p_1, p_2, p_3, p_4)$,

$$
f(h_t) =
\begin{cases}
p_k & \text{con } k = \min\{\,i : p_i \text{ non ancora eseguito in } h_t\,\}\\[4pt]
\text{final} & \text{se tutti i } p_i \text{ sono stati eseguiti}
\end{cases}
$$

```text
get_metrics:web-01 già eseguito?        ── no ──► get_metrics(web-01)        [il frontend è sovraccarico?]
          │ sì
read_log:web-01 già eseguito?           ── no ──► read_log(web-01)           [errori lato web?]
          │ sì
read_log:api-01 già eseguito?           ── no ──► read_log(api-01)           [scendo al livello successivo]
          │ sì
get_database_connections:db-01 già?     ── no ──► get_database_connections   [verifico l'ipotesi DB]
          │ sì
          └──► FINAL ANSWER (sintesi)
```

Il percorso imita il ragionamento di un sistemista: si parte dal punto in cui l'utente percepisce il problema (il frontend) e si **scende lungo la catena delle dipendenze** fino alla causa.

La chiave `"tool:server"` (es. `"read_log:api-01"`) identifica univocamente un'osservazione e serve a evitare di ripetere la stessa chiamata (*deduplicazione*).

> ⚠️ **Limiti voluti (da discutere nel debrief).**
> 1. Il planner guarda **quali** tool sono stati chiamati, **non cosa hanno restituito**: l'ordine è fisso. È un *workflow travestito da agente*.
> 2. La risposta finale è **testo scritto a mano**: il valore "99 connessioni su 100" non è calcolato dalle osservazioni. Se si cambia lo scenario, la diagnosi resta identica (si verifica nella sez. 14).
> 3. Non c'è gestione degli errori: se un tool fallisce il planner prosegue come se nulla fosse.
>
> È proprio *questa* funzione che un LLM sostituisce: un modello legge i risultati, adatta il piano e sintetizza la risposta a partire dall'evidenza.

In [8]:
# @dataclass genera automaticamente __init__, __repr__ e __eq__ a partire dagli attributi annotati.

@dataclass
class ToolCall:
    # Rappresenta un'azione a_t di tipo "chiamata a tool"
    name: str                   # nome del tool (chiave nel TOOL_REGISTRY)
    arguments: dict[str, Any]   # argomenti nominati, passati come **kwargs alla funzione


@dataclass
class AgentState:
    # Stato / memoria di lavoro dell'agente: corrisponde alla storia h_t
    user_request: str           # richiesta iniziale dell'utente (u)
    # Lista delle osservazioni raccolte. default_factory=list crea una lista
    # NUOVA per ogni istanza: un default mutabile condiviso (= []) sarebbe un bug
    # classico, e infatti dataclass lo vieta sollevando ValueError.
    observations: list[dict[str, Any]] = field(default_factory=list)
    # Risposta finale; None finché l'agente non ha concluso
    final_answer: str | None = None


def simulated_planner(state: AgentState) -> ToolCall | str:
    """A deterministic planner that mimics a simple agent reasoning path."""
    # Policy f(h_t): restituisce la prossima ToolCall oppure una stringa (risposta finale).

    # List comprehension: estrae da ogni osservazione la chiave "tool:server"
    # (es. "get_metrics:web-01"). .get evita KeyError se la chiave mancasse.
    called = [obs.get("tool") for obs in state.observations]

    # Passo 1: parto dal frontend, dove l'utente percepisce la lentezza.
    # Il frontend è sovraccarico? -> metriche di web-01
    if "get_metrics:web-01" not in called:
        return ToolCall("get_metrics", {"server": "web-01"})

    # Passo 2: ci sono errori lato web? -> log di web-01
    if "read_log:web-01" not in called:
        return ToolCall("read_log", {"server": "web-01", "last_lines": 5})

    # Passo 3: il frontend sembra sano, scendo al livello successivo -> log del backend
    if "read_log:api-01" not in called:
        return ToolCall("read_log", {"server": "api-01", "last_lines": 5})

    # Passo 4: i log API parlano di attese verso il DB -> verifico il pool di connessioni
    if "get_database_connections:db-01" not in called:
        return ToolCall("get_database_connections", {"server": "db-01"})

    # Sintesi finale basata sulle osservazioni raccolte
    # NB: il testo è FISSO (hard-coded), non derivato dai valori osservati:
    # è il limite principale del planner simulato (vedi sez. 14).
    # Le stringhe adiacenti tra parentesi vengono concatenate automaticamente da Python.
    return (
        "Causa più probabile: saturazione del pool di connessioni al database. "
        "Il frontend non mostra anomalie evidenti; i log API indicano attese e timeout verso il database; "
        "il database ha 99 connessioni attive su 100, pari al 99% di utilizzo. "
        "Consiglio: aumentare temporaneamente il pool o liberare connessioni bloccate, previa verifica umana."
    )

## 8. Agent loop manuale

Questo è il cuore concettuale del modulo: il **ciclo di controllo** che trasforma un "decisore" (planner o LLM) in un agente.

### Pseudocodice

```text
state ← AgentState(user_request)
for turn = 1 … T_max:                         # budget: numero massimo di turni
    decision ← policy(state)                  # DECIDE
    if decision è una stringa:                # azione "final"
        state.final_answer ← decision
        return state                          # terminazione normale
    result ← execute(decision)                # ACT  (dispatch via registry)
    state.observations.append(result)         # OBSERVE → aggiorna h_t
state.final_answer ← "MAX_TURNS raggiunto"    # terminazione per budget
return state
```

### Due condizioni di terminazione

| Condizione | Significato | Esito |
|---|---|---|
| la policy restituisce una `str` | l'agente ritiene di avere abbastanza evidenza | risposta finale |
| `turn > max_turns` | budget esaurito | diagnosi incompleta (*graceful degradation*) |

Il budget `max_turns` è un **guardrail** essenziale: con un LLM reale nulla garantisce che il modello smetta di chiamare tool. Senza limite si rischiano loop infiniti, costi e latenze fuori controllo. Il numero di chiamate ai tool è limitato superiormente:

$$
N_{\text{tool}} \le T_{\max} - 1 \quad \text{se l'agente conclude}, \qquad N_{\text{tool}} = T_{\max} \quad \text{se esaurisce il budget}
$$

### Separazione delle responsabilità

```text
run_manual_agent  (orchestrazione: turni, budget, stato, logging)
      │
      ├──► simulated_planner(state)      "cosa fare"       (sostituibile con un LLM)
      │
      └──► execute_tool_call(call)       "come farlo"      (dispatch + allowlist)
                  │
                  └──► TOOL_REGISTRY[name](**arguments)
```

### Struttura di un'osservazione salvata nello stato

```python
{
    "tool": "read_log:api-01",                          # chiave di deduplicazione
    "arguments": {"server": "api-01", "last_lines": 5}, # cosa è stato chiesto
    "result": {...}                                     # cosa è stato osservato
}
```

**Output atteso:** 4 turni di tool call (web-01 metriche, web-01 log, api-01 log, db-01 connessioni) e la risposta finale al 5° turno.

> ⚠️ **Nota tecnica.** `execute_tool_call` gestisce il nome sconosciuto (`UNKNOWN_TOOL`) ma non le **eccezioni** sollevate dalla funzione, ad esempio un `TypeError` se il modello passa un argomento non previsto. In un sistema reale conviene racchiudere la chiamata in `try/except` e restituire un errore strutturato (`{"success": False, "error": "TOOL_EXCEPTION", ...}`), coerentemente con il principio *errori come dati*.

In [9]:
def execute_tool_call(call: ToolCall) -> dict[str, Any]:
    # ACT: traduce una decisione (ToolCall) in un'esecuzione reale.

    # Allowlist: se il nome non è registrato NON si esegue nulla e si restituisce
    # un errore strutturato, che diventa un'osservazione per l'agente.
    if call.name not in TOOL_REGISTRY:
        return {"success": False, "error": "UNKNOWN_TOOL", "tool": call.name}

    # Lookup nel registry: nome -> oggetto funzione
    func = TOOL_REGISTRY[call.name]
    # Unpacking del dizionario come argomenti nominati:
    # {"server": "api-01"}  ->  func(server="api-01")
    result = func(**call.arguments)
    return result


def run_manual_agent(user_request: str, max_turns: int = 8) -> AgentState:
    # Inizializza la storia h_1 con la sola richiesta dell'utente
    state = AgentState(user_request=user_request)

    # Ciclo di controllo con budget: turn va da 1 a max_turns incluso
    for turn in range(1, max_turns + 1):
        # DECIDE: la policy restituisce una ToolCall oppure una stringa finale
        decision = simulated_planner(state)

        # Terminazione normale: la decisione è la risposta finale
        if isinstance(decision, str):
            state.final_answer = decision
            print(f"\nFINAL ANSWER:\n{decision}")
            return state   # uscita anticipata dal ciclo e dalla funzione

        # Logging dell'azione scelta (osservabilità in tempo reale)
        print(f"TURN {turn}: calling {decision.name}({decision.arguments})")
        # ACT: esecuzione del tool tramite dispatch
        result = execute_tool_call(decision)
        print("RESULT:", result)

        # OBSERVE: aggiornamento dello stato, h_{t+1} = h_t ⊕ (a_t, o_t)
        # Il server (se presente) entra nella chiave di deduplicazione "tool:server";
        # .get(..., "") evita KeyError per tool senza argomento server.
        key_server = decision.arguments.get("server", "")
        state.observations.append({
            "tool": f"{decision.name}:{key_server}",   # es. "read_log:api-01"
            "arguments": decision.arguments,           # azione a_t
            "result": result,                          # osservazione o_t
        })

    # Uscita dal for senza risposta finale: budget esaurito.
    # Si restituisce comunque uno stato valido con un messaggio esplicito.
    state.final_answer = "MAX_TURNS raggiunto: diagnosi incompleta."
    print(state.final_answer)
    return state

# Esecuzione dell'agente sulla segnalazione dell'utente.
# Lo stato finale viene salvato in `state` per l'analisi della trace (sez. 9).
state = run_manual_agent("Il portale aziendale è diventato molto lento negli ultimi dieci minuti.")

TURN 1: calling get_metrics({'server': 'web-01'})
RESULT: {'success': True, 'server': 'web-01', 'cpu': 39, 'memory': 61}
TURN 2: calling read_log({'server': 'web-01', 'last_lines': 5})
RESULT: {'success': True, 'server': 'web-01', 'logs': ['12:01 INFO static asset served', '12:02 INFO request completed', '12:03 INFO healthcheck ok']}
TURN 3: calling read_log({'server': 'api-01', 'last_lines': 5})
RESULT: {'success': True, 'server': 'api-01', 'logs': ['12:01 INFO request accepted', '12:02 WARNING database request waiting', '12:03 WARNING request timeout', '12:04 WARNING database request waiting']}
TURN 4: calling get_database_connections({'server': 'db-01'})
RESULT: {'success': True, 'server': 'db-01', 'active': 99, 'max': 100, 'utilization': 0.99}

FINAL ANSWER:
Causa più probabile: saturazione del pool di connessioni al database. Il frontend non mostra anomalie evidenti; i log API indicano attese e timeout verso il database; il database ha 99 connessioni attive su 100, pari al 99% di 

## 9. Trace

Un sistema agentico deve essere **osservabile**: dobbiamo poter ricostruire *perché* ha preso una decisione. Poiché la sequenza di azioni non è scritta dal programmatore ma scelta a runtime, la trace è l'unico modo per fare debugging, audit e valutazione.

La trace è la storia finale:

$$
h_T = \big(u,\ (a_1, o_1),\ (a_2, o_2),\ \dots,\ (a_{T-1}, o_{T-1}),\ \text{final}\big)
$$

Leggendola si verifica che la conclusione sia **supportata dall'evidenza** (*grounding*): ogni affermazione della risposta finale dovrebbe corrispondere a un'osservazione.

### Cosa contiene una trace in produzione

| Campo | Perché serve |
|---|---|
| `trace_id` / `span_id` | correlare tutte le chiamate di una stessa esecuzione |
| timestamp e latenza per passo | individuare colli di bottiglia |
| modello e versione del prompt | riproducibilità, confronto tra versioni |
| token in input/output | controllo dei costi |
| tool, argomenti, risultato | ricostruire il ragionamento |
| approvazioni umane (chi, quando) | accountability sulle azioni con side effect |

La nostra `print_trace` è la versione minimale di questo concetto. L'Agents SDK (sez. 13) include un sistema di tracing integrato; in alternativa si usano standard come OpenTelemetry.

In [10]:
def print_trace(state: AgentState) -> None:
    # Stampa leggibile della storia h_T: richiesta -> sequenza (azione, osservazione) -> risposta

    # 1) La richiesta iniziale dell'utente
    print("USER REQUEST:")
    print(state.user_request)

    # 2) Le osservazioni in ordine cronologico.
    #    enumerate(..., start=1) fornisce un contatore che parte da 1 invece che da 0.
    print("\nTRACE:")
    for i, obs in enumerate(state.observations, start=1):
        print(f"{i}. {obs['tool']} -> {obs['result']}")

    # 3) La conclusione dell'agente
    print("\nFINAL:")
    print(state.final_answer)

# Analisi della trace dell'esecuzione della sezione 8
print_trace(state)

USER REQUEST:
Il portale aziendale è diventato molto lento negli ultimi dieci minuti.

TRACE:
1. get_metrics:web-01 -> {'success': True, 'server': 'web-01', 'cpu': 39, 'memory': 61}
2. read_log:web-01 -> {'success': True, 'server': 'web-01', 'logs': ['12:01 INFO static asset served', '12:02 INFO request completed', '12:03 INFO healthcheck ok']}
3. read_log:api-01 -> {'success': True, 'server': 'api-01', 'logs': ['12:01 INFO request accepted', '12:02 WARNING database request waiting', '12:03 WARNING request timeout', '12:04 WARNING database request waiting']}
4. get_database_connections:db-01 -> {'success': True, 'server': 'db-01', 'active': 99, 'max': 100, 'utilization': 0.99}

FINAL:
Causa più probabile: saturazione del pool di connessioni al database. Il frontend non mostra anomalie evidenti; i log API indicano attese e timeout verso il database; il database ha 99 connessioni attive su 100, pari al 99% di utilizzo. Consiglio: aumentare temporaneamente il pool o liberare connessioni b

## 10. Calcolo dell'evidenza

Nel nostro caso l'indizio quantitativo principale è:

$$
\text{UtilizzoConnessioni} = \frac{\text{active}}{\text{max}} = \frac{99}{100} = 0{,}99 \;\;(99\%)
$$

### Perché un utilizzo del 99% produce timeout (intuizione dalla teoria delle code)

Un pool di connessioni si comporta come un sistema di servizio con coda. Detto $\rho$ il fattore di utilizzo, per il modello di coda più semplice (M/M/1) il tempo medio di permanenza nel sistema è:

$$
W = \frac{1}{\mu - \lambda} = \frac{1/\mu}{1 - \rho}, \qquad \rho = \frac{\lambda}{\mu}
$$

dove $\lambda$ è il tasso di arrivo delle richieste e $\mu$ il tasso di servizio. Il fattore $\frac{1}{1-\rho}$ **esplode** quando $\rho \to 1$:

| $\rho$ | 0,50 | 0,80 | 0,90 | 0,95 | 0,99 |
|---|---|---|---|---|---|
| $\frac{1}{1-\rho}$ (moltiplicatore del tempo di servizio) | 2× | 5× | 10× | 20× | **100×** |

Un pool reale è un sistema multi-servente e l'utilizzo delle connessioni non coincide esattamente con $\rho$, ma l'intuizione resta valida: **vicino alla saturazione i tempi di attesa crescono in modo non lineare**, ed è per questo che il sistema "diventa lento" senza che CPU e RAM mostrino anomalie.

### Catena causale ricostruita

```text
pool DB saturo (99/100) ──► richieste API in coda ("database request waiting")
                        ──► superamento del timeout ("request timeout")
                        ──► portale lento per l'utente
```

### Triangolazione delle evidenze

| Fonte | Osservazione | Supporta l'ipotesi "pool DB saturo"? |
|---|---|---|
| metriche `web-01` | CPU 39%, RAM 61% | esclude il frontend come causa |
| log `web-01` | solo INFO | esclude errori lato web |
| log `api-01` | attese verso il DB, timeout | **sì** (sintomo compatibile) |
| connessioni `db-01` | 99 / 100 | **sì** (evidenza quantitativa diretta) |
| log `db-01` | *database healthy* | no, ma il log non misura il pool: evidenza debole |

Una buona diagnosi combina **evidenza qualitativa** (log) ed **evidenza quantitativa** (metriche) provenienti da fonti diverse.

In [11]:
# Rilettura dell'evidenza chiave direttamente dal tool (nessun side effect)
conn = get_database_connections("db-01")
# Il tool restituisce già "utilization" come frazione (0.99): qui la ricalcoliamo
# esplicitamente ed esprimiamo il valore in percentuale, per mostrare la formula.
utilization_pct = conn["active"] / conn["max"] * 100
# Format spec ":.1f" -> numero decimale con una cifra dopo la virgola (99.0)
print(f"Utilizzo connessioni DB: {utilization_pct:.1f}%")

Utilizzo connessioni DB: 99.0%


## 11. Human-in-the-loop

Ora mostriamo un'azione con side effect. Il sistema **non deve** riavviare direttamente un servizio senza approvazione.

Il pattern *human-in-the-loop* (HITL) separa **chi propone** un'azione (l'agente) da **chi la autorizza** (un operatore umano):

```text
 Agente propone restart ──► tool risponde "requires_approval"
                                        │
                                        ▼
                            Operatore umano valuta
                             ├── approva ──► esecuzione con approved=True ──► side effect + log
                             └── rifiuta ──► nessuna azione, esito registrato
```

Il ruolo dell'umano non è solo "premere OK": deve valutare **se l'azione è appropriata**. Qui c'è uno spunto importante: la diagnosi indica il **database** come causa, ma l'azione proposta riavvia l'**API**. Il riavvio agirebbe su un sintomo, non sulla causa radice, e potrebbe interrompere le richieste in corso.

> ⚠️ Nel notebook l'approvazione è simulata con una variabile (`human_approval = True`). In un sistema reale la decisione arriverebbe da un canale **esterno al modello** (interfaccia utente, ticket, messaggio di chat) e verrebbe registrata con identità dell'approvatore e timestamp.

> Nota: l'esecuzione approvata **modifica lo stato globale** (aggiunge una riga ai log di `api-01`). Rieseguendo le sezioni precedenti senza reset si otterrebbero log diversi.

In [12]:
# FASE 1 — PROPOSTA: l'agente chiede il riavvio SENZA approvazione.
# Risultato atteso: {"success": False, "requires_approval": True, "message": ...}
# Nessun side effect: lo stato di `servers` resta invariato.
proposal = restart_service("api-01", "api", approved=False)
proposal

{'success': False,
 'requires_approval': True,
 'message': 'Restart of api on api-01 requires approval.'}

In [13]:
# Simuliamo l'approvazione umana.
# FASE 2 — DECISIONE: in un sistema reale questo valore arriverebbe da un
# operatore (UI, ticket, chat), NON dal modello.
human_approval = True

if human_approval:
    # FASE 3a — ESECUZIONE AUTORIZZATA: approved=True supera il gate.
    # SIDE EFFECT: servizio impostato a "running" + nuova riga nei log di api-01.
    action_result = restart_service("api-01", "api", approved=True)
else:
    # FASE 3b — RIFIUTO: nessuna chiamata al tool, si registra solo l'esito
    action_result = {"success": False, "message": "Operazione non approvata dall'operatore umano."}

# Visualizza l'esito dell'azione
action_result

{'success': True, 'server': 'api-01', 'service': 'api', 'status': 'restarted'}

## 12. Opzionale: OpenAI Responses API con function calling

> La cella seguente richiede:
>
> ```bash
> export OPENAI_API_KEY=...
> export OPENAI_MODEL=gpt-5.6
> ```
> 
> 
> (il nome del modello è un valore configurabile: verificare che sia disponibile per il proprio account prima della lezione).
>
> Il codice è volutamente separato dal laboratorio principale, che rimane eseguibile anche senza API key.

### Stesso loop, decisore diverso

L'idea chiave: **il loop è quello della sezione 8**, cambia solo chi decide.

| Loop manuale (sez. 8) | Responses API (questa sezione) |
|---|---|
| `simulated_planner(state)` | `client.responses.create(...)` |
| `ToolCall(name, arguments)` | item di output con `type == "function_call"` (`name`, `arguments` come stringa JSON, `call_id`) |
| `execute_tool_call(call)` | `TOOL_REGISTRY[call.name](**args)` |
| osservazione in `state.observations` | item `function_call_output` collegato tramite `call_id` |
| `state` ($h_t$) | lista `input_items` (l'intera conversazione) |
| risposta `str` → stop | nessuna `function_call` nell'output → stop |
| `max_turns` | `max_turns` |

### Sequenza di un turno

```text
 Python                                           Modello (API)
   │  input_items + instructions + tools_schema       │
   │ ───────────────────────────────────────────────► │
   │                                                  │ decide
   │ ◄─────────────────────────────────────────────── │
   │  output: [function_call(name, arguments, call_id)]
   │
   │  esegue TOOL_REGISTRY[name](**args)
   │  aggiunge function_call_output(call_id, output)
   │
   │  input_items (storia aggiornata)                  │
   │ ───────────────────────────────────────────────► │ ... fino a una risposta senza tool call
```

Dettagli importanti:

- La API è usata in modalità **stateless**: ad ogni turno si reinvia l'intera storia `input_items`. (In alternativa l'API consente di far conservare lo stato al server, collegando le risposte con `previous_response_id`.)
- `response.output` contiene tutti gli item prodotti dal modello (messaggi, eventuali item di *reasoning*, tool call): vanno reinseriti nella storia **tutti**, convertiti in dizionari con `model_dump(exclude_none=True)`.
- `call_id` accoppia ogni risultato alla tool call che l'ha generato: il modello può chiedere **più tool call nello stesso turno** (chiamate parallele).
- `instructions` è il *system prompt*: definisce ruolo e regole (*raccogli evidenza prima di concludere*, *non inventare metriche*).

> ⚠️ **Note tecniche.**
> 1. **Schema strict**: vedi sez. 6. Lo schema è ora conforme: tutte le proprietà sono incluse in `required` e `additionalProperties` è `false`.
> 2. **Istruzioni ≠ enforcement**: nella demo API utilizziamo `tools_schema_safe`, che esclude `restart_service`. È più robusto del chiedere al modello, tramite prompt, di non auto-approvare un'azione con side effect.
> 3. **Robustezza**: a differenza di `execute_tool_call`, qui l'accesso `TOOL_REGISTRY[call.name]` non controlla i nomi sconosciuti (possibile `KeyError`) e `json.loads` / la chiamata al tool non sono protetti da `try/except`.
> 4. I log dei server entrano nel contesto del modello: sono **input non fidato** (rischio di *prompt injection* se un log contenesse istruzioni).


In [ ]:
# Carica le variabili presenti nel file .env
# Per default cerca ".env" nella directory corrente e nelle directory superiori.
load_dotenv()

def run_openai_responses_agent(user_request: str, max_turns: int = 8):

    # Guardia: senza API key la demo viene saltata senza errori,
    # così il notebook resta eseguibile da cima a fondo.
    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY non impostata: salto la demo API.")
        return None

    # Import "lazy": la libreria openai serve solo qui, non è una dipendenza del laboratorio base
    from openai import OpenAI

    # Il client legge automaticamente OPENAI_API_KEY dall'ambiente
    client = OpenAI()
    # Modello configurabile via variabile d'ambiente, con valore di fallback
    model = os.getenv("OPENAI_MODEL", "gpt-5.6")

    # System prompt: ruolo dell'agente e regole di comportamento (guardrail "soft", testuali)
    instructions = """
    You are an IT operations diagnostic agent.
    Gather evidence before making conclusions.
    Prefer read-only tools.
    Do not restart services unless explicitly approved.
    Do not invent metrics or logs.
    """

    # Storia h_1: la conversazione inizia con il solo messaggio dell'utente
    input_items = [{"role": "user", "content": user_request}]

    # Agent loop con budget (stessa struttura della sez. 8)
    for turn in range(max_turns):
        # DECIDE: il modello riceve storia + istruzioni + catalogo dei tool
        response = client.responses.create(
            model=model,
            instructions=instructions,
            input=input_items,
            tools=tools_schema_safe,   # catalogo strict e read-only della sez. 6
        )

        # Aggiunge alla storia TUTTI gli item prodotti dal modello (inclusi i function_call
        # e gli eventuali item di reasoning), convertiti da oggetti Pydantic a dict.
        # exclude_none=True omette i campi valorizzati a None, che non vanno reinviati.
        input_items.extend(item.model_dump(exclude_none=True) for item in response.output)

        # Filtra le richieste di esecuzione tool (possono essere più d'una per turno)
        tool_calls = [item for item in response.output if item.type == "function_call"]

        # Condizione di terminazione: nessuna tool call -> il modello ha risposto
        if not tool_calls:
            # output_text: proprietà di comodo che concatena il testo dei messaggi di output
            print(response.output_text)
            return response.output_text

        # ACT + OBSERVE per ogni tool call richiesta
        for call in tool_calls:
            # Gli argomenti arrivano come STRINGA JSON: vanno deserializzati in un dict
            args = json.loads(call.arguments)
            # Dispatch tramite registry (senza controllo sui nomi sconosciuti: vedi nota)
            result = TOOL_REGISTRY[call.name](**args)
            print(f"TOOL: {call.name}({args}) -> {result}")
            # Il risultato torna al modello come item "function_call_output",
            # collegato alla richiesta tramite call_id e serializzato in JSON (stringa)
            input_items.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result),
            })

    # Budget esaurito senza risposta finale
    print("MAX_TURNS raggiunto")
    return None

# Chiamata commentata: decommentare per eseguire la demo (richiede API key e rete)
run_openai_responses_agent("Il portale aziendale è diventato molto lento negli ultimi dieci minuti.")


TOOL: get_server_status({'server': 'web-01'}) -> {'success': True, 'server': 'web-01', 'role': 'frontend', 'status': 'online', 'services': {'nginx': 'running'}}
TOOL: get_metrics({'server': 'web-01'}) -> {'success': True, 'server': 'web-01', 'cpu': 39, 'memory': 61}
TOOL: read_log({'server': 'web-01', 'last_lines': 10}) -> {'success': True, 'server': 'web-01', 'logs': ['12:01 INFO static asset served', '12:02 INFO request completed', '12:03 INFO healthcheck ok']}
TOOL: get_server_status({'server': 'app-01'}) -> {'success': False, 'error': 'SERVER_NOT_FOUND', 'server': 'app-01'}
TOOL: get_metrics({'server': 'app-01'}) -> {'success': False, 'error': 'SERVER_NOT_FOUND', 'server': 'app-01'}
TOOL: read_log({'server': 'app-01', 'last_lines': 10}) -> {'success': False, 'error': 'SERVER_NOT_FOUND', 'server': 'app-01'}
TOOL: get_server_status({'server': 'db-01'}) -> {'success': True, 'server': 'db-01', 'role': 'database', 'status': 'online', 'services': {'mysql': 'running'}}
TOOL: get_metrics({

'**Probabile causa individuata:** saturazione delle connessioni al database.\n\n- `web-01`: operativo, nginx attivo, CPU 39%, RAM 61%, nessun errore recente.\n- `db-01`: operativo, MySQL attivo, CPU 52%, RAM 69%.\n- **Connessioni DB: 99 attive su 100 massime (99%)**.\n\nLa lentezza è quindi verosimilmente dovuta alle richieste in attesa di una connessione al database, non a carenza di CPU o memoria. Non ho riavviato né modificato alcun servizio.\n\nProssimi interventi consigliati: identificare il client che mantiene troppe connessioni, verificare eventuali connessioni bloccate o non rilasciate e controllare il pool applicativo prima di valutare un aumento del limite.'

## 13. Opzionale: OpenAI Agents SDK

Il vantaggio dell'SDK è che gestisce il ciclo:

```text
model → tool → model → final output
```

Noi lo introduciamo solo **dopo** averlo implementato manualmente: così è chiaro che cosa l'astrazione nasconde.

### Cosa automatizza l'SDK rispetto alla sez. 12

| Responsabilità | Sez. 12 (manuale) | Agents SDK |
|---|---|---|
| Loop e terminazione | `for turn in range(max_turns)` scritto a mano | `Runner.run_sync(..., max_turns=8)` |
| Schema dei tool | `tools_schema` scritto a mano | generato da **type hint e docstring** della funzione decorata |
| Dispatch | `TOOL_REGISTRY[...]` | automatico |
| Serializzazione argomenti/risultati | `json.loads` / `json.dumps` | automatica |
| Superamento del budget | messaggio `"MAX_TURNS raggiunto"` | eccezione `MaxTurnsExceeded` |
| Tracing | `print` | tracing integrato |

**Scelta di design da evidenziare:** l'agente riceve solo **tre tool read-only**. `restart_service` **non è esposto**: è l'applicazione più forte del principio di minimo privilegio, perché un'azione che il modello non può chiamare non può essere chiamata per errore o per manipolazione.

I tool sono **wrapper** delle funzioni già scritte: si riusa l'implementazione e si espone un'interfaccia pensata per il modello (nomi e docstring diversi).

> ⚠️ **Nota tecnica — nome del decoratore (verificato con `openai-agents` 0.22).** Nell'SDK il decoratore per i tool si chiama **`function_tool`**. Il nome `tool` corrisponde invece al **sottomodulo** `agents.tool`: l'istruzione `from agents import Agent, Runner, tool` quindi *non fallisce* (importa il modulo), ma alla riga `@tool` si ottiene `TypeError: 'module' object is not callable`. L'errore avviene **fuori** dal blocco `try/except`, quindi la cella si interrompe. Correzione:
>
> ```python
> from agents import Agent, Runner, function_tool
>
> @function_tool
> def server_metrics(server: str) -> dict:
>     ...
> ```

> ⚠️ **Nota tecnica — Jupyter.** `Runner.run_sync` non funziona se esiste già un event loop attivo, come in Jupyter (lo dice la documentazione del metodo). In un notebook si usa la versione asincrona direttamente in una cella: `result = await Runner.run(agent, "...", max_turns=8)`.

> Nota: l'`Agent` non specifica `model=`, quindi usa il modello predefinito dell'SDK e ignora `OPENAI_MODEL`; per allinearlo alla sez. 12 si può passare `model=os.getenv("OPENAI_MODEL")`.

In [ ]:
def run_agents_sdk_demo():
    # Guardia sulla API key, come nella sez. 12
    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY non impostata: salto la demo Agents SDK.")
        return None

    # Import "lazy" protetto: se il pacchetto openai-agents non è installato
    # si stampa un messaggio di aiuto invece di interrompere il notebook.
    # (Vedi Nota tecnica: il decoratore corretto è `function_tool`.)
    try:
        from agents import Agent, Runner, tool
    except Exception as exc:
        print("Installa openai-agents per eseguire questa demo:")
        print("pip install openai-agents")
        print(exc)
        return None

    # --- Definizione dei tool esposti all'agente -----------------------------
    # Il decoratore registra la funzione come tool: nome, type hint e docstring
    # vengono usati per generare automaticamente lo schema JSON per il modello.
    # Sono wrapper sottili delle funzioni read-only della sez. 3.

    @tool
    def server_metrics(server: str) -> dict:
        """Return CPU and memory utilization for a server."""
        return get_metrics(server)

    @tool
    def server_logs(server: str, last_lines: int = 5) -> dict:
        """Return latest log entries for a server."""
        return read_log(server, last_lines)

    @tool
    def database_connections(server: str = "db-01") -> dict:
        """Return database connection utilization."""
        return get_database_connections(server)

    # --- Definizione dell'agente = Model + Tools + istruzioni --------------------
    agent = Agent(
        name="IT Operations Agent",           # nome leggibile (usato anche nel tracing)
        instructions=(                        # system prompt
            "Diagnose infrastructure problems using tools. "
            "Gather evidence before conclusions. Never invent metrics or logs."
        ),
        # Solo tool read-only: restart_service NON è esposto (minimo privilegio)
        tools=[server_metrics, server_logs, database_connections],
    )

    # --- Esecuzione: il Runner implementa l'agent loop ---------------------------
    # run_sync = versione sincrona (bloccante); max_turns = budget del loop.
    # (In Jupyter usare `await Runner.run(...)`: vedi Nota tecnica.)
    result = Runner.run_sync(
        agent,
        "Il portale aziendale è molto lento negli ultimi dieci minuti. Diagnostica il problema.",
        max_turns=8,
    )

    # final_output: la risposta finale dell'agente
    print(result.final_output)
    return result

# Chiamata commentata: decommentare per eseguire la demo (richiede API key e openai-agents)
run_agents_sdk_demo()

## 14. Challenge

Modificare lo scenario in modo che:

- `web-01` abbia CPU normale;
- `api-01` mostri timeout verso DB;
- `db-01` abbia saturazione delle connessioni.

Obiettivo: verificare se l'agente conclude che il collo di bottiglia più plausibile è il database.

### Come funziona il reset

`reset_environment()` ricrea `servers` da `INITIAL_SERVERS` con una nuova `deepcopy`, annullando i side effect della sez. 11. La parola chiave `global` è necessaria: senza di essa l'assegnazione `servers = ...` creerebbe una variabile **locale** alla funzione e lo stato globale resterebbe invariato. I tool leggono `servers` al momento della chiamata, quindi vedono subito il nuovo oggetto.

### Spunto critico per l'aula

Lo scenario richiesto dalla challenge **coincide con quello iniziale**, e il planner simulato è deterministico con risposta finale fissa: la cella produce quindi la **stessa trace** della sez. 8, e la "verifica" è superata per costruzione. Per rendere la challenge significativa si possono proporre varianti che mettono in luce i limiti del planner:

| Variante (modifica prima di `run_manual_agent`) | Cosa dovrebbe concludere un agente reale | Cosa fa il planner simulato |
|---|---|---|
| `servers["db-01"]["connections"]["active"] = 20` | il DB **non** è saturo: cercare altrove | dichiara comunque "99 connessioni su 100" |
| `servers["web-01"]["cpu"] = 98` e log di errore su nginx | il collo di bottiglia è il frontend | ignora il valore e scende comunque verso il DB |
| `del servers["db-01"]` | il DB non è raggiungibile: problema diverso | riceve `SERVER_NOT_FOUND` ma conclude lo stesso |

**Esercizio di estensione:** riscrivere la parte finale del planner in modo che la risposta sia **derivata dalle osservazioni**. Traccia:

```python
db = next(o["result"] for o in state.observations
          if o["tool"] == "get_database_connections:db-01")
if db.get("success") and db["utilization"] >= 0.9:
    return f"Pool DB saturo: {db['active']}/{db['max']} ({db['utilization']:.0%})."
return "Nessuna saturazione del DB rilevata: servono altre verifiche."
```

Poi eseguire le tre varianti e confrontare le trace con la versione originale.

In [17]:
def reset_environment():
    # `global` indica che l'assegnazione riguarda la variabile a livello di modulo,
    # non una nuova variabile locale alla funzione.
    global servers
    # Nuova copia profonda della fixture: annulla tutti i side effect precedenti
    # (es. la riga di log aggiunta dal riavvio approvato nella sez. 11)
    servers = deepcopy(INITIAL_SERVERS)
    return servers

# 1) Ripristino dello stato iniziale
reset_environment()
# 2) Nuova esecuzione dell'agente con una segnalazione diversa (timeout intermittenti)
challenge_state = run_manual_agent("Gli utenti segnalano timeout intermittenti sul sito, ma CPU e RAM sembrano normali.")
# 3) Analisi della trace: confrontarla con quella della sezione 9
print_trace(challenge_state)

TURN 1: calling get_metrics({'server': 'web-01'})
RESULT: {'success': True, 'server': 'web-01', 'cpu': 39, 'memory': 61}
TURN 2: calling read_log({'server': 'web-01', 'last_lines': 5})
RESULT: {'success': True, 'server': 'web-01', 'logs': ['12:01 INFO static asset served', '12:02 INFO request completed', '12:03 INFO healthcheck ok']}
TURN 3: calling read_log({'server': 'api-01', 'last_lines': 5})
RESULT: {'success': True, 'server': 'api-01', 'logs': ['12:01 INFO request accepted', '12:02 WARNING database request waiting', '12:03 WARNING request timeout', '12:04 WARNING database request waiting']}
TURN 4: calling get_database_connections({'server': 'db-01'})
RESULT: {'success': True, 'server': 'db-01', 'active': 99, 'max': 100, 'utilization': 0.99}

FINAL ANSWER:
Causa più probabile: saturazione del pool di connessioni al database. Il frontend non mostra anomalie evidenti; i log API indicano attese e timeout verso il database; il database ha 99 connessioni attive su 100, pari al 99% di 

## 15. Debrief

Domande da discutere in aula.

**1. Quale parte del comportamento era codificata dal programmatore?**

**2. Quale parte è stata lasciata alla decisione dell'agente?**

**3. Perché il riavvio non deve essere automatico?**

**4. Perché un tool fallito può essere utile come osservazione?**

**5. In quali casi usereste un workflow classico invece di un agente?**

Regola pratica: *usare la soluzione più semplice che funziona*, e introdurre autonomia solo dove porta un vantaggio misurabile.

## Riepilogo dei concetti

| Concetto | Definizione sintetica | Sezione |
|---|---|---|
| Agent | Model + Tools + State + ControlLoop | 1 |
| Policy $\pi(a \mid h_t)$ | criterio con cui si sceglie la prossima azione | 1, 7 |
| Environment | sistema osservato e modificato tramite tool | 2 |
| Tool | contratto: argomenti tipizzati → dict strutturato | 3 |
| Errori come dati | i fallimenti diventano osservazioni, non eccezioni | 3 |
| Side effect | azione che modifica lo stato del mondo | 4 |
| Fail-closed | in assenza di approvazione l'azione non avviene | 4 |
| Tool registry | allowlist e dispatch nome → funzione | 5 |
| Tool schema | interfaccia JSON Schema vista dal modello | 6 |
| Agent loop | Decide → Act → Observe, con budget e terminazione | 8 |
| Trace | storia $h_T$ usata per debugging, audit e valutazione | 9 |
| Grounding | la conclusione è supportata da evidenze osservate | 9, 10 |
| Human-in-the-loop | l'umano autorizza le azioni rischiose | 11 |
| Minimo privilegio | esporre al modello solo i tool strettamente necessari | 4, 13 |